In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import os
import sys
sys.path.append('.')

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
import pickle
from datetime import datetime
import logging
from sklearn.linear_model import LinearRegression

import torch
from torch import nn
from torch.utils.data import Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn.functional as F

import FinanceDataReader as fdr
import yfinance as yf

SEED=10
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

In [2]:
datafile = './NASDAQ100_m.csv'
data_list = pd.read_csv(datafile)

In [3]:
stock_list = data_list.columns[1:]
tmp = []
for s in stock_list:
    tmp.append(s.split('_')[0])
l = tmp

In [4]:
class FeatureEngineering:
    def __init__(self, stock_name):
        self.stock_name = stock_name

        #단순 이동 평균
    def SMA(self,  data, column='Close', period=30):
        return data[column].rolling(period).mean()

    #지수 이동 평균
    def EMA(self, data, period=20, column='Close'):
        return data[column].ewm(span=period, adjust=False).mean()
    
    def Bollingerband(self, data, period=20, column='Close'):
        data[column+'_SMA'] = self.SMA(data, column=column, period=period)
        data[column+'_UB'] = data[column+'_SMA'] + 2*data[column+'_SMA'].rolling(20).std()
        data[column+'_LB'] = data[column+'_SMA'] - 2*data[column+'_SMA'].rolling(20).std()
        
        return data

    def MACD(self, data, period_long=26, period_short=12, period_signal=9, column='Close'):
        data[column+'_EMA'] = self.EMA(data, period=20 ,column=column)
        
        data[column+'_ShortEMA'] = self.EMA(data, period_short, column=column)

        data[column+'_LongEMA'] = self.EMA(data, period_long, column=column)

        data[column+'_MACD'] = data[column+'_ShortEMA']- data[column+'_LongEMA']

              #signal
        data[column+'_Signal_Line'] = self.EMA(data, period_signal, column=column+'_MACD')

        return data
    
    def Momentum(self, data, period=7, column='Close'):
        data[column+'_7D'] = data[column].shift(period)
        data[column+'_1D'] = data[column].shift(1)
        data[column+'_Momentum'] = data[column+'_1D'] / data[column+'_7D'] - 1
        
        return data
    
    def RSI(self, data, period=14, column='Close'):
        delta = data[column].diff(1)
        delta = delta.dropna()

        up = delta.copy()
        down = delta.copy()
        up[up<0] = 0
        down[down>0] = 0
        data[column+'_up'] = up
        data[column+'_down'] = down

        AVG_Gain = self.SMA(data, period=period, column=column+'_up')
        AVG_Loss = abs(self.SMA(data, period=period, column=column+'_down'))
        RS = AVG_Gain / AVG_Loss

        RSI = 100.0 - (100.0/(1.0+RS))
        data[column+'_RSI'] = RSI
  
        return data

    def get_data(self, train):
        #print('Feature Engineering...')
        
        self.train = train
        c = self.stock_name
        
        #train
        #print("\nTrain dataset")
        self.train = self.Bollingerband(self.train, column=c)
        self.train = self.MACD(self.train, column=c)
        self.train = self.Momentum(self.train, column=c)
        self.train = self.RSI(self.train, column=c)
        
        #self.train.dropna(inplace=True)
        #self.train.reset_index(inplace=True)
        #self.train.drop(columns=['Date'], inplace=True)
        
        #print("Done!")
        
        return self.train

In [22]:
import requests
import pandas_market_calendars as mcal

def extract_historical_csvdata(symbol):
    api_key = '5fb4dbfbc7da14.06701815'
    url = f'https://eodhistoricaldata.com/api/intraday/{symbol}.US?&api_token={api_key}&interval=1m'
    df = pd.read_csv(url).iloc[:,2:]
    
    if df.empty == False:
        df['Datetime'] = df['Datetime']+"+00:00"
        df = df.set_index('Datetime')

        #xkrx = mcal.get_calendar('NYSE')
        #early = xkrx.schedule(start_date='2023-01-01', end_date='2023-03-08')

        df.index = pd.to_datetime(df.index)
        #df = df.resample('5min').first().loc[mcal.date_range(early, frequency='1min')].interpolate(method='linear', limit_direction='both')
    
    return df

In [6]:
for code in tqdm(l[:-1]): 
    print(code)
    data = extract_historical_csvdata(code)[:-1]
    if data.empty == False:
        fe = FeatureEngineering('Close')
        data = fe.get_data(data)

        data['weekday'] = pd.DatetimeIndex(data.index).weekday
        data['year'] = pd.DatetimeIndex(data.index).year
        data['month'] = pd.DatetimeIndex(data.index).month
        data['day'] = pd.DatetimeIndex(data.index).day

        data.to_csv(f'/data3/jihyeon/StockDiffusion/data/{code}.csv', index=0)  

#data.isna().sum()

  0%|                                                   | 0/101 [00:00<?, ?it/s]

ATVI


  1%|▍                                          | 1/101 [00:04<06:53,  4.14s/it]

ADBE


  2%|▊                                          | 2/101 [00:07<06:15,  3.80s/it]

...

XLNX


 99%|████████████████████████████████████████▌| 100/101 [06:16<00:03,  3.00s/it]

close


 99%|████████████████████████████████████████▌| 100/101 [06:17<00:03,  3.77s/it]


HTTPError: HTTP Error 404: Not Found

In [27]:
import requests
import pandas_market_calendars as mcal

def extract_historical_csvdata(symbol):
    api_key = '5fb4dbfbc7da14.06701815'
    url = f'https://eodhistoricaldata.com/api/intraday/{symbol}.US?&api_token={api_key}&interval=1m'
    df = pd.read_csv(url).iloc[:,2:]
    
    if df.empty == False:
        df['Datetime'] = df['Datetime']+"+00:00"
        df = df.set_index('Datetime')

        #xkrx = mcal.get_calendar('NYSE')
        #early = xkrx.schedule(start_date='2023-01-01', end_date='2023-03-08')

        df.index = pd.to_datetime(df.index)
        #df = df.resample('5min').first().loc[mcal.date_range(early, frequency='1min')].interpolate(method='linear', limit_direction='both')
    
    return df

In [28]:
for code in tqdm(['DJIA', 'QQQ', 'SPY']): 
    print(code)
    data = extract_historical_csvdata(code)[:-1]
    if data.empty == False:
        fe = FeatureEngineering('Close')
        data = fe.get_data(data)

        data['weekday'] = pd.DatetimeIndex(data.index).weekday
        data['year'] = pd.DatetimeIndex(data.index).year
        data['month'] = pd.DatetimeIndex(data.index).month
        data['day'] = pd.DatetimeIndex(data.index).day

        data.to_csv(f'/data3/jihyeon/StockDiffusion/intraday/{code}.csv', index=0)  

  0%|                                                     | 0/3 [00:00<?, ?it/s]

DJIA


 33%|███████████████                              | 1/3 [00:01<00:02,  1.48s/it]

QQQ


 67%|██████████████████████████████               | 2/3 [00:09<00:05,  5.55s/it]

SPY


100%|█████████████████████████████████████████████| 3/3 [00:17<00:00,  5.76s/it]
